In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import re

In [13]:
import pandas as pd
import re
import os


def summarize_error_counts(file_paths, n_words=5):
    """
    Build a DataFrame with:
      - column 1: error (first n_words of error message)
      - column 2: error_example (one full example message)
      - columns 3+: counts of that error type per input file
    """
    def first_n_words(s, n):
        s = re.sub(r"\s+", " ", str(s).strip())
        if not s:
            return ""
        parts = s.split(" ")
        return " ".join(parts[:n])

    def summarize_errors_for_file(path, source_name):
        df = pd.read_csv(path)
        if "error" not in df.columns:
            raise KeyError(f"'error' column not found in {path}")

        err = df["error"].astype("string").fillna("").str.strip()
        has_error = err.ne("")

        df_errors = df.loc[has_error].copy()
        df_errors["error_type"] = df_errors["error"].apply(
            lambda s: first_n_words(s, n_words)
        )
        df_errors["source"] = source_name

        summary = (
            df_errors
            .groupby(["source", "error_type"])
            .agg(
                example_error=("error", "first"),
                count=("error_type", "size")
            )
            .reset_index()
        )
        return summary

    all_summaries = []
    for path in file_paths:
        source_name = os.path.basename(path)
        all_summaries.append(summarize_errors_for_file(path, source_name))

    combined = pd.concat(all_summaries, ignore_index=True)

    # Wide counts table: one column per source
    counts_wide = combined.pivot_table(
        index="error_type",
        columns="source",
        values="count",
        fill_value=0,
        aggfunc="sum",
    )

    # One example per error_type
    examples = (
        combined
        .sort_values("source")
        .groupby("error_type", as_index=True)["example_error"]
        .first()
    )

    # Build final DataFrame
    final_df = counts_wide.copy()
    final_df.insert(0, "error_example", examples)
    final_df.insert(0, "error", final_df.index)
    final_df = final_df.reset_index(drop=True)

    # Add total row with sum of all numeric columns
    numeric_cols = final_df.select_dtypes(include=["number"]).columns
    totals = final_df[numeric_cols].sum()

    total_row = {col: (totals[col] if col in totals.index else "") for col in final_df.columns}
    total_row["error"] = "__TOTAL__"
    total_row["error_example"] = ""

    final_df = pd.concat([final_df, pd.DataFrame([total_row])], ignore_index=True)

    return final_df


In [14]:
files = ['./results_local_trend_1000_seed_42_fixed_params_clipped.csv', './results_seasonal_uc_1000_seed_42_daily_clipped.csv', './results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_600_reworked.csv']
summary_df = summarize_error_counts(files)
display(summary_df)


,error,error_example,results_local_trend_1000_seed_42_fixed_params_clipped.csv,results_seasonal_uc_1000_seed_42_daily_clipped.csv,results_seasonal_uc_1000_seed_42_daily_clipped_tree_timeout_600_reworked.csv
0,All three periodicities are not,All three periodicities are not within 10% ran...,0,19,76
1,All tree periodicities are not,All tree periodicities are not within 10% rang...,16,0,0
2,DataFrame passed is too short,DataFrame passed is too short len < 2,60,61,51
3,Failed to fit initial training,Failed to fit initial training model: Unable t...,0,8,0
4,Failed to fit second training,Failed to fit second training model: Unable to...,0,11,0
5,Failed to generate predictions: Unable,Failed to generate predictions: Unable to allo...,0,1,0
6,Insufficient data in one of,Insufficient data in one of the segments for m...,2,2,0
7,No time difference calculated.,No time difference calculated.,0,0,58
8,Not enough data. Do not,Not enough data. Do not have at least 100 days...,0,0,70
9,Not enough data. Start time,Not enough data. Start time needed for train 2...,71,71,0
